<a href="https://colab.research.google.com/github/ihmorol/bvp-boundary-pinn/blob/main/campaign-readiness/tools/colab_main_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Main run on Colab: reference, PINN campaign, evaluation

This notebook runs the whole main experiment of the five-field boundary problem from start to finish and keeps everything in your Google Drive.

**How to use it**

1. *Runtime > Change runtime type*: pick a GPU (T4 or better) if you have one. CPU works too, only slower.
2. Check the settings cell below. The defaults are the committed campaign settings; you do not need to change anything.
3. *Runtime > Run all*. Allow Drive access when asked.
4. If Colab disconnects, open the notebook again and *Run all* again. Finished steps and finished runs are skipped; an unfinished run starts again from its beginning. Use one Colab session at a time per Drive folder.

**What it does, in order:** connects Drive and makes the results folder, checks the GPU, clones the repository at a fixed commit, installs the exact package versions from `uv.lock` in their own environment, then runs `tools/main_run.py`: tests, the classical reference from scratch (with its error bars), a one-minute dry run of the whole chain on your device, a timing measurement with a run-time estimate, the campaign (4 arms x 5 seeds), and the evaluation (tables, figures, metrics). Logs of every step are in the `logs` folder in Drive.

The full campaign takes many hours (the timing step prints an estimate for your machine). One Colab session will usually not finish it: run it again in a new session until the last step says *All steps finished*.

The 22 physical values, Eqs. 14 to 18 and B1 to B12 are fixed and cannot be set here (AGENTS.md section 3).

In [1]:
# ============================ SETTINGS: everything the run needs is here ============================
# Where the results go, inside "My Drive". The folder is made if it does not exist. Use a new folder for a
# new campaign; reuse the same folder to resume.
DRIVE_FOLDER = "bvp-boundary-pinn/main-run-P4"

# The code. REPO_URL is the repository's own address (git remote "origin").
REPO_URL = "https://github.com/ihmorol/bvp-boundary-pinn.git"
REPO_BRANCH = "impl/campaign-readiness"
REPO_COMMIT = ""   # "" = the commit this Drive folder started with, or the branch tip on the very first start

# Hardware.
DEVICE = "auto"    # "auto" (GPU if Colab gives one), "cuda" or "cpu"
JOBS = "auto"      # runs at once: "auto" = 1 on a GPU, the CPU count (at most 4) on CPU

# What to run. None = what configs/pinn.json says (arms B0 to B3, seeds 0 to 4, campaign "P4").
ARMS = None
SEEDS = None
CAMPAIGN_NAME = None

# Training settings, written into configs/pinn.json ("common" block). These are the committed values.
# Changing one makes new run folders; a Drive folder started with other values refuses to continue.
PINN_SETTINGS = {
    "n_points": 2000,       # training points on [0, 20], clustered at the wall
    "n_monitor": 1000,      # separate monitor points (never trained on)
    "width": 40,            # neurons per hidden layer
    "depth": 4,             # hidden layers
    "adam_steps": 20000,    # Adam steps in total (split over the stages of B2 and B3)
    "lr": 0.001,            # Adam learning rate at the start
    "lr_end": 0.0001,       # and at the end (exponential decay)
    "lbfgs_iters": 5000,    # L-BFGS iterations (log scale)
    "lm_iters": 260,        # Levenberg-Marquardt steps (balanced scale)
    "windows": [0.0, 0.5, 2.0, 6.0, 20.0],  # window edges of B1, B2, B3
}

# Checks around the campaign.
RUN_TESTS = True          # pytest and the record check before anything else
SMOKE_TEST = True         # one-minute dry run of every arm on this device before the real runs
MEASURE_TIMING = True     # measure step times here and print the run-time estimate
CLASSICAL_REPEATS = 5     # repeats of each solve_bvp timing (figure FX1, table TX1)

# Software, exact versions. Every other package comes pinned from uv.lock.
PYTHON_VERSION = "3.12.13"
UV_VERSION = "0.10.9"
TORCH_VERSION = "2.15.0"  # GPU: the PyPI build (CUDA 13.2); CPU: 2.15.0+cpu, as in uv.lock
# ======================================== END OF SETTINGS ========================================

In [2]:
# Step A: GPU check, Google Drive, results folder.
import json, os, pathlib, shutil, subprocess, sys, time

# CPU runtimes have no nvidia-smi at all.
q = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True) if shutil.which("nvidia-smi") else None
HAS_GPU = q is not None and q.returncode == 0 and bool(q.stdout.strip())
print("GPU (name, driver, memory):", q.stdout.strip() if HAS_GPU else
      "none. For a GPU: Runtime > Change runtime type > GPU, then Run all again.")
if DEVICE == "cuda" and not HAS_GPU:
    raise RuntimeError("DEVICE is 'cuda' but this runtime has no GPU. Pick a GPU runtime or set DEVICE = 'auto'.")
WANT_GPU = HAS_GPU and DEVICE != "cpu"

from google.colab import drive
drive.mount("/content/drive")
OUT = pathlib.Path("/content/drive/MyDrive") / DRIVE_FOLDER
OUT.mkdir(parents=True, exist_ok=True)
print("Results folder:", OUT)

GPU (name, driver, memory): Tesla T4, 580.82.07, 15360 MiB
Mounted at /content/drive
Results folder: /content/drive/MyDrive/bvp-boundary-pinn/main-run-P4


In [3]:
# Step B: the repository at a fixed commit.
REPO = pathlib.Path("/content/bvp-boundary-pinn")

def token():
    # Only needed if the repository is private: put a GitHub token in Colab's Secrets (key icon) as GITHUB_TOKEN.
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def git(*args, secret=None, cwd=REPO):
    shown = " ".join(args) if not secret else " ".join(args).replace(secret, "***")
    print("$ git", shown)
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).replace(secret, "***") if secret else (r.stderr or r.stdout)
        raise RuntimeError(f"git {shown} failed: {msg.strip()}")
    return r.stdout.strip()

def fetch_url():
    t = token()
    return (REPO_URL.replace("https://", f"https://x-access-token:{t}@"), t) if t else (REPO_URL, None)

if not (REPO / ".git").exists():
    try:
        git("clone", "--quiet", REPO_URL, str(REPO), cwd="/content")
    except RuntimeError as err:
        url, t = fetch_url()
        if not t:
            raise RuntimeError(f"{err}\nIf the repository is private, add a GitHub token as the Colab secret GITHUB_TOKEN.")
        git("clone", "--quiet", url, str(REPO), secret=t, cwd="/content")
        git("remote", "set-url", "origin", REPO_URL)  # do not keep the token in .git/config

url, t = fetch_url()
git("fetch", "--quiet", url, f"+refs/heads/{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}", secret=t)
start = OUT / "steps" / "start.json"
commit = REPO_COMMIT or (json.loads(start.read_text())["commit"] if start.exists() else "")
if commit:
    print("Using commit", commit, "(from", "REPO_COMMIT)" if REPO_COMMIT else "the first session of this Drive folder)")
    git("checkout", "--quiet", "--force", commit)
else:
    git("checkout", "--quiet", "--force", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}")
git("clean", "-fdq", "--", "configs", "src", "tools", "tests", "engine")
print("Commit:", git("log", "-1", "--format=%H %s"))

$ git clone --quiet https://github.com/ihmorol/bvp-boundary-pinn.git /content/bvp-boundary-pinn
$ git fetch --quiet https://github.com/ihmorol/bvp-boundary-pinn.git +refs/heads/impl/campaign-readiness:refs/remotes/origin/impl/campaign-readiness
Using commit e04e00d9dc13f8b894ab34cb266af18815f53bd3 (from the first session of this Drive folder)
$ git checkout --quiet --force e04e00d9dc13f8b894ab34cb266af18815f53bd3
$ git clean -fdq -- configs src tools tests engine
$ git log -1 --format=%H %s
Commit: e04e00d9dc13f8b894ab34cb266af18815f53bd3 Keep Colab's kernel variables out of the run environment


In [4]:
# Step C: Python and the exact packages, in their own environment (/content/venv).
VENV = pathlib.Path("/content/venv")
PY = VENV / "bin" / "python"

# Colab sets these for its own Python 3.13 kernel. Inherited by our Python 3.12 environment they break it:
# MPLBACKEND names an inline backend that is not installed there (matplotlib refuses to import), and
# PYTHONPATH can pull in Colab's own packages. Every command below runs without them.
VENV_ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "MPLBACKEND")}
VENV_ENV.update({"MPLBACKEND": "Agg", "PYTHONNOUSERSITE": "1", "PYTHONUNBUFFERED": "1"})

def sh(cmd, cwd=None):
    # Run a command, show its output here (Colab hides output written by subprocesses), stop on failure.
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=VENV_ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-30:]
    if proc.wait():
        raise RuntimeError(f"command failed (exit code {proc.returncode}): {' '.join(map(str, cmd))}\n" + "\n".join(tail))

def venv_torch():
    r = subprocess.run([str(PY), "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
                       capture_output=True, text=True, env=VENV_ENV)
    if r.returncode and PY.exists():
        print("PyTorch does not import yet:", (r.stderr or r.stdout).strip().splitlines()[-1:] or "")
    return r.stdout.split() if r.returncode == 0 else None

marker = VENV / "installed.json"
wanted = {"python": PYTHON_VERSION, "torch": TORCH_VERSION, "gpu": WANT_GPU,
          "lock": subprocess.run(["sha256sum", str(REPO / "uv.lock")], capture_output=True, text=True).stdout.split()[0]}
if marker.exists() and json.loads(marker.read_text()).get("wanted") == wanted and venv_torch():
    print("Environment already installed:", venv_torch())
else:
    sh([sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_VERSION}"])
    sh(["uv", "venv", "--clear", "--python", PYTHON_VERSION, str(VENV)])  # uv downloads this Python if needed
    req = subprocess.run(["uv", "export", "--frozen", "--all-groups", "--no-hashes", "--no-emit-project",
                          "--no-header", "--format", "requirements-txt"],
                         cwd=REPO, capture_output=True, text=True, check=True, env=VENV_ENV).stdout
    keep = [l for l in req.splitlines() if l.strip() and not l.lstrip().startswith(("#", "torch=="))]
    (VENV / "requirements.txt").write_text("\n".join(keep) + "\n")
    sh(["uv", "pip", "install", "--python", PY, "-r", VENV / "requirements.txt"])
    if WANT_GPU:
        sh(["uv", "pip", "install", "--python", PY, f"torch=={TORCH_VERSION}"])
        ok = venv_torch()
        if not ok or ok[1] != "True":
            # The CUDA 13.2 build needs a recent NVIDIA driver. If Colab's driver is older, use the PyTorch
            # build Colab itself ships for this GPU (recorded in environment-*.json), so the GPU still works.
            colab = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                                   capture_output=True, text=True).stdout.strip()
            base, _, tag = colab.partition("+")
            print(f"\nWARNING: torch {TORCH_VERSION} cannot use this GPU (driver too old?). "
                  f"Falling back to Colab's own build, torch {colab}.\n")
            index = f"https://download.pytorch.org/whl/{tag}" if tag.startswith("cu") else "https://pypi.org/simple"
            sh(["uv", "pip", "install", "--python", PY, f"torch=={base}", "--index-url", index,
                "--extra-index-url", "https://pypi.org/simple", "--index-strategy", "unsafe-best-match"])
    else:
        sh(["uv", "pip", "install", "--python", PY, "--no-deps", f"torch=={TORCH_VERSION}+cpu",
            "--index-url", "https://download.pytorch.org/whl/cpu"])
    if not venv_torch():
        raise RuntimeError("PyTorch does not import in the new environment; see the messages above.")
    if WANT_GPU and venv_torch()[1] != "True":
        raise RuntimeError("This runtime has a GPU, but no PyTorch build here can use it. Set DEVICE = 'cpu' to run "
                           "on the CPU, or pick another GPU runtime, then Run all again.")
    marker.write_text(json.dumps({"wanted": wanted, "installed": venv_torch()}))
print("torch, GPU usable:", venv_torch())
sh([PY, "-c", "import numpy, scipy, matplotlib, sympy, pytest, sys; "
    "print('python', sys.version.split()[0], 'numpy', numpy.__version__, 'scipy', scipy.__version__, "
    "'matplotlib', matplotlib.__version__, 'sympy', sympy.__version__, 'pytest', pytest.__version__)"])

$ /usr/bin/python3 -m pip install -q uv==0.10.9
$ uv venv --clear --python 3.12.13 /content/venv
Using CPython 3.12.13
Creating virtual environment at: venv
Activate with: source venv/bin/activate
$ uv pip install --python /content/venv/bin/python -r /content/venv/requirements.txt
Using Python 3.12.13 environment at: venv
Resolved 25 packages in 226ms
Prepared 25 packages in 1.72s
Installed 25 packages in 119ms
 + contourpy==1.4.0
 + cycler==0.12.1
 + filelock==4.2.12
 + fonttools==4.68.0
 + fsspec==2026.9.0
 + iniconfig==2.3.1
 + jinja2==3.1.6
 + kiwisolver==1.5.1
 + markupsafe==3.0.4
 + matplotlib==3.11.2
 + mpmath==1.3.0
 + networkx==3.7
 + numpy==2.5.5
 + packaging==26.3
 + pillow==13.0.0
 + pluggy==1.7.0
 + pygments==2.21.0
 + pyparsing==3.3.3
 + pytest==9.2.0
 + python-dateutil==4.0.0
 + scipy==1.18.1
 + setuptools==84.2.1
 + sympy==1.14.0
 + typing-extensions==4.16.0
 + tzdata==2026.6
$ uv pip install --python /content/venv/bin/python torch==2.15.0
Using Python 3.12.13 environme

In [ ]:
# Step D: the main run. Safe to run again after a disconnect: finished steps and runs are skipped.
settings = {"out_dir": str(OUT), "device": DEVICE, "jobs": JOBS, "arms": ARMS, "seeds": SEEDS,
            "campaign_name": CAMPAIGN_NAME, "pinn_common": PINN_SETTINGS, "run_tests": RUN_TESTS,
            "smoke_test": SMOKE_TEST, "measure_timing": MEASURE_TIMING, "classical_repeats": CLASSICAL_REPEATS}
(OUT / "run_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
import signal
proc = subprocess.Popen([str(PY), "tools/main_run.py", "--settings", str(OUT / "run_settings.json")], cwd=REPO,
                        env=VENV_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                        start_new_session=True)  # VENV_ENV from step C: no Colab PYTHONPATH or MPLBACKEND
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    # Stop the whole process tree, so no training keeps running in the background. The run in progress
    # has no meta.json yet, so the next Run all trains it again from its start.
    os.killpg(proc.pid, signal.SIGTERM)
    proc.wait()
    raise
if proc.wait():
    raise RuntimeError(f"The main run stopped with an error (exit code {proc.returncode}). The message above names the "
                     f"step; its full log is in {OUT / 'logs'}. Fix the cause, then Run all again to resume.")

Main run, output folder /content/drive/MyDrive/bvp-boundary-pinn/main-run-P4
Device: cuda (Tesla T4); GPU available: True; runs at once: 1; CPU count: 2
Environment recorded in /content/drive/MyDrive/bvp-boundary-pinn/main-run-P4/environment-2026-11-27T124529.json



== [1/7] tests: already finished (2026-11-27T12:38:48), skipped

== [2/7] reference: started 2026-11-27T12:45:30
   $ /content/venv/bin/python -m bvp_pinn.reference.run
/content/bvp-boundary-pinn/src/bvp_pinn/reference/equations.py:68: RuntimeWarning: overflow encountered in power
  + Nr / Pr * (A**3 * th1 / q + 3 * c * A**2 * th1**2 + A**3 * th2)
/content/bvp-boundary-pinn/src/bvp_pinn/reference/equations.py:68: RuntimeWarning: overflow encountered in multiply
  + Nr / Pr * (A**3 * th1 / q + 3 * c * A**2 * th1**2 + A**3 * th2)
/content/bvp-boundary-pinn/src/bvp_pinn/reference/equations.py:68: RuntimeWarning: invalid value encountered in add
  + Nr / Pr * (A**3 * th1 / q + 3 * c * A**2 * th1**2 + A**3 * th2)
/content/bvp-b

In [ ]:
# Step E: a first look at the results (everything is also in Drive).
from IPython.display import Image, Markdown, display
name = CAMPAIGN_NAME or json.loads((REPO / "configs" / "pinn.json").read_text())["campaign"]["name"]
res = OUT / "results" / name
for f in ("accuracy.md", "wall_values.md", "cost.md", "failures.md"):
    if (res / f).exists():
        display(Markdown((res / f).read_text()))
for f in ("FB1_profiles", "FB2_errors", "FX1_error_vs_time"):
    if (res / "figures" / f"{f}.png").exists():
        display(Image(str(res / "figures" / f"{f}.png"), width=900))
print("All results:", res)